<a href="https://colab.research.google.com/github/MiguelUTEC26/Parcia04_Miguel_Angel_Diaz_Lopez_25-4356-2019/blob/main/agrupacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from itertools import combinations

In [2]:
# 1. Cargar datos
ruta_archivo = "https://raw.githubusercontent.com/MiguelUTEC26/Parcia04_Miguel_Angel_Diaz_Lopez_25-4356-2019/refs/heads/main/data/raw/clave_F_agrupacion.csv"
df = pd.read_csv(ruta_archivo)

In [5]:
df = df.fillna(0)

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.isnull().sum()

,0
registro_id,0
edad,0
ingresos,0
frecuencia_uso,0
gasto_promedio,0
satisfaccion,0
reclamos,0
antiguedad_meses,0


In [9]:
df.describe()

,edad,ingresos,frecuencia_uso,gasto_promedio,satisfaccion,reclamos,antiguedad_meses
count,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000,250.000000
mean,39.244000,1063.948000,5.435960,84.714200,6.844720,2.408000,18.332000
std,10.016877,429.301685,3.053921,49.368022,1.994396,1.986282,13.076458
min,18.000000,300.000000,0.000000,5.000000,0.000000,0.000000,1.000000
25%,32.000000,747.250000,2.727500,46.652500,5.172500,1.000000,9.000000
50%,39.000000,983.500000,5.475000,72.315000,7.270000,2.000000,16.000000
75%,47.000000,1342.250000,7.832500,119.715000,8.467500,4.000000,28.000000
max,66.000000,2900.000000,13.360000,260.000000,10.000000,12.000000,56.000000


In [11]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   registro_id       250 non-null    object 
 1   edad              250 non-null    int64  
 2   ingresos          250 non-null    int64  
 3   frecuencia_uso    250 non-null    float64
 4   gasto_promedio    250 non-null    float64
 5   satisfaccion      250 non-null    float64
 6   reclamos          250 non-null    int64  
 7   antiguedad_meses  250 non-null    int64  
dtypes: float64(3), int64(4), object(1)
memory usage: 15.8+ KB
None


In [12]:
variables_numericas = ["edad", "ingresos", "reclamos","antiguedad_meses"]
for columna in variables_numericas:
    media = df[columna].mean()
    desviacion = df[columna].std()

    limite_inferior = media - 2 * desviacion
    limite_superior = media + 2 * desviacion

    atipicos = df[
        (df[columna] < limite_inferior) |
        (df[columna] > limite_superior)
    ]

    print("Variable:", columna)
    print("Media:", round(media, 2))
    print("Desviación estándar:", round(desviacion, 2))
    print("Límite inferior:", round(limite_inferior, 2))
    print("Límite superior:", round(limite_superior, 2))
    print("Cantidad de valores atípicos:", len(atipicos))


Variable: edad
Media: 39.24
Desviación estándar: 10.02
Límite inferior: 19.21
Límite superior: 59.28
Cantidad de valores atípicos: 7
Variable: ingresos
Media: 1063.95
Desviación estándar: 429.3
Límite inferior: 205.34
Límite superior: 1922.55
Cantidad de valores atípicos: 8
Variable: reclamos
Media: 2.41
Desviación estándar: 1.99
Límite inferior: -1.56
Límite superior: 6.38
Cantidad de valores atípicos: 5
Variable: antiguedad_meses
Media: 18.33
Desviación estándar: 13.08
Límite inferior: -7.82
Límite superior: 44.48
Cantidad de valores atípicos: 6


In [13]:
# 4. Variables del modelo
X = df[["edad", "ingresos", "reclamos","antiguedad_meses"]]


In [15]:
# 5. K-Means con K=2
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

modelo_k2 = KMeans(n_clusters=2, random_state=0, n_init=10)
df["cluster_k2"] = modelo_k2.fit_predict(X)
print(pd.DataFrame(modelo_k2.cluster_centers_, columns=X.columns))
print("Silhouette K=2:", silhouette_score(X, df["cluster_k2"]))


        edad     ingresos  reclamos  antiguedad_meses
0  35.284848   810.375758  3.200000         12.581818
1  46.929412  1556.176471  0.870588         29.494118
Silhouette K=2: 0.6086182601648069


In [16]:
# 6. K-Means con K=4
modelo_k4 = KMeans(n_clusters=4, random_state=0, n_init=10)
df["cluster_k4"] = modelo_k4.fit_predict(X)
centroides_k4 = pd.DataFrame(modelo_k4.cluster_centers_, columns=X.columns)
print(centroides_k4)
print(df["cluster_k4"].value_counts())
print("Silhouette K=4:", silhouette_score(X, df["cluster_k4"]))


        edad     ingresos  reclamos  antiguedad_meses
0  38.242105  1027.200000  2.705263         14.842105
1  51.666667  1985.444444  0.722222         33.888889
2  32.404762   634.333333  3.500000          9.928571
3  47.660377  1497.754717  0.716981         32.622642
cluster_k4
0    95
2    84
3    53
1    18
Name: count, dtype: int64
Silhouette K=4: 0.5473244092294336


In [20]:
# 7. Identificar clúster de mayor riesgo
centroides_k4["riesgo"] = centroides_k4["edad"] + centroides_k4["ingresos"] + + centroides_k4["antiguedad_meses"]
cluster_riesgo = centroides_k4["riesgo"].idxmax()
print("Clúster de mayor riesgo:", cluster_riesgo)


Clúster de mayor riesgo: 1


In [23]:
# 8. Filtrar personas del clúster de riesgo
personas_riesgo = df[df["cluster_k4"] == cluster_riesgo]
print(personas_riesgo[["edad", "ingresos", "antiguedad_meses"]].describe())
personas_riesgo.to_csv("personas_riesgo_cluster_kmeans.csv", index=False)


            edad     ingresos  antiguedad_meses
count  18.000000    18.000000         18.000000
mean   51.666667  1985.444444         33.888889
std     6.471658   256.800271         10.571265
min    40.000000  1752.000000          9.000000
25%    47.250000  1863.750000         29.250000
50%    51.000000  1909.500000         34.500000
75%    54.000000  2027.500000         40.500000
max    66.000000  2900.000000         52.000000
